In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load files
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
incident_start = pd.to_datetime("2024-02-20 23:34:32", utc=True)
incident_end   = pd.to_datetime("2024-02-21 00:04:32", utc=True)

# Helper to compute grouped summary for files with a 'value' column
def grouped_summary(df, group_keys, value_col='value', top_n=50):
    if df.shape[0] == 0:
        cols = group_keys + ['count', 'min_ts', 'max_ts', 'p95']
        return pd.DataFrame(columns=cols)
    # global p95 per group (using the entire series for that group)
    p95 = df.groupby(group_keys)[value_col].quantile(0.95).reset_index().rename(columns={value_col: 'p95'})
    agg = df.groupby(group_keys).agg(
        count=(value_col, 'size'),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max')
    ).reset_index()
    summary = agg.merge(p95, on=group_keys)
    summary = summary.sort_values('count', ascending=False).head(top_n)
    # keep columns compact and ensure timestamp dtype
    summary['min_ts'] = pd.to_datetime(summary['min_ts'], utc=True)
    summary['max_ts'] = pd.to_datetime(summary['max_ts'], utc=True)
    return summary

# 1) metric.csv summary
metric_summary = grouped_summary(metric, ['cmdb_id', 'kpi_name'], value_col='value', top_n=50)
metric_total_count = int(metric.shape[0])
metric_window_count = int(metric[(metric['timestamp'] >= incident_start) & (metric['timestamp'] <= incident_end)].shape[0])

# 2) trace.csv summary
trace_summary = grouped_summary(trace, ['cmdb_id', 'trace_name'], value_col='value', top_n=50)
trace_total_count = int(trace.shape[0])
trace_window_count = int(trace[(trace['timestamp'] >= incident_start) & (trace['timestamp'] <= incident_end)].shape[0])

# 3) log.csv summary
log_summary = grouped_summary(log, ['cmdb_id', 'log_name'], value_col='value', top_n=50)
log_total_count = int(log.shape[0])
log_window_count = int(log[(log['timestamp'] >= incident_start) & (log['timestamp'] <= incident_end)].shape[0])

# 4) error_logs.csv aggregates and samples
error_total_count = int(error_logs.shape[0])
if error_total_count > 0:
    error_min_ts = error_logs['timestamp'].min()
    error_max_ts = error_logs['timestamp'].max()
    error_window_count = int(error_logs[(error_logs['timestamp'] >= incident_start) & (error_logs['timestamp'] <= incident_end)].shape[0])
    top_cmdb_by_msgs = (error_logs.groupby('cmdb_id')
                        .size()
                        .reset_index(name='message_count')
                        .sort_values('message_count', ascending=False)
                        .head(10))
    # sample up to 5 messages overall (show timestamp, cmdb_id, message)
    error_samples = error_logs[['timestamp', 'cmdb_id', 'message']].sort_values('timestamp').head(5)
else:
    error_min_ts = pd.NaT
    error_max_ts = pd.NaT
    error_window_count = 0
    top_cmdb_by_msgs = pd.DataFrame(columns=['cmdb_id','message_count'])
    error_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Keep outputs compact: show summaries (already limited), totals and error summaries
metric_summary, metric_total_count, metric_window_count, \
trace_summary, trace_total_count, trace_window_count, \
log_summary, log_total_count, log_window_count, \
error_total_count, error_min_ts, error_max_ts, error_window_count, top_cmdb_by_msgs, error_samples

```
Out[1]:
```
```python
# Summarize previous execution results in plain English as a single string.
# Uses variables already loaded in the kernel (metric, trace, log, error_logs, and the computed counts).

top_metric_services = metric['cmdb_id'].value_counts().head(3).index.tolist() if 'metric' in globals() and not metric.empty else []
top_trace_services  = trace['cmdb_id'].value_counts().head(3).index.tolist()  if 'trace' in globals()  and not trace.empty  else []
top_log_services    = log['cmdb_id'].value_counts().head(3).index.tolist()    if 'log' in globals()    and not log.empty    else []

summary = (
    f"Summary of telemetry (UTC incident window 2024-02-20 23:34:32 to 2024-02-21 00:04:32):\n"
    f"- metric.csv: total rows = {metric_total_count}, rows inside window = {metric_window_count} "
    f"(all metric rows fall inside the incident window). Top metric-producing services (top 3): {top_metric_services}.\n"
    f"- trace.csv: total rows = {trace_total_count}, rows inside window = {trace_window_count} "
    f"(all trace rows fall inside the incident window). Top trace-producing services (top 3): {top_trace_services}.\n"
    f"- log.csv: total rows = {log_total_count}, rows inside window = {log_window_count} "
    f"(all log rows fall inside the incident window). Top log-producing services (top 3): {top_log_services}.\n"
    f"- error_logs.csv: total rows = {error_total_count}. No error log messages recorded (no samples to show).\n\n"
    f"Notable observations from the aggregated tables shown earlier:\n"
    f"  * metric groups include common KPIs such as cpu, mem, diskio, latency-50/90 with P95 values reported per group.\n"
    f"  * trace groups show heavy activity from checkoutservice (duration_mean, duration_p95, error_rate, row_count).\n"
    f"  * log groups include log.row_count and log.error_count for many services; error_count values shown were 0 in the aggregates.\n\n"
    f"Conclusion: telemetry coverage for metrics, traces, and logs fully overlaps the incident window, and there are no recorded error_log entries to inspect further."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count                    min_ts                    max_ts           p95
0               adservice         cpu     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  4.034705e+00
1               adservice      diskio     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  1.109214e+05
2               adservice  latency-50     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  3.838967e-03
3               adservice  latency-90     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  7.600920e-03
4               adservice         mem     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  1.873336e+08
..                    ...         ...    ...                       ...                       ...           ...
45         paymentservice         mem     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  4.965425e+07
46         paymentservice      socket     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  5.860000e+00
47         paymentservice    workload     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  1.198683e+00
48  productcatalogservice         cpu     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  1.986661e+01
49  productcatalogservice      diskio     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  8.459010e+09

[50 rows x 6 columns], 1772, 1772,             cmdb_id                                trace_name  count                    min_ts                    max_ts         p95
0   checkoutservice  trace.from_checkoutservice.duration_mean     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    0.021787
1   checkoutservice   trace.from_checkoutservice.duration_p95     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    0.088514
2   checkoutservice     trace.from_checkoutservice.error_rate     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  302.600000
4   checkoutservice  trace.from_frontendservice.duration_mean     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    0.213623
..              ...                                       ...    ...                       ...                       ...         ...
45     emailservice   trace.from_checkoutservice.duration_p95     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    0.000698
46     emailservice     trace.from_checkoutservice.error_rate     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    0.000000
47     emailservice      trace.from_checkoutservice.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00   35.000000
48     emailservice             trace.from_root.duration_mean     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    0.000054
49     emailservice              trace.from_root.duration_p95     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    0.000088

[50 rows x 6 columns], 3800, 3800,                   cmdb_id         log_name  count                    min_ts                    max_ts     p95
0               adservice  log.error_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00     0.0
1               adservice    log.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00   460.6
2             cartservice  log.error_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00     0.0
3             cartservice    log.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00   814.4
4         checkoutservice  log.error_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00     0.0
5         checkoutservice    log.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00   105.0
6         currencyservice  log.error_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00     0.0
7         currencyservice    log.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  2543.4
8            emailservice  log.error_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00     0.0
9            emailservice    log.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    35.0
10               frontend  log.error_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00     0.0
11               frontend    log.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00  2826.4
12         paymentservice  log.error_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00     0.0
13         paymentservice    log.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00    70.0
14  recommendationservice  log.error_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00     0.0
15  recommendationservice    log.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00   590.6
19        shippingservice    log.row_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00   513.6
18        shippingservice  log.error_count     25 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00     0.0
17                  redis    log.row_count      5 2024-02-20 23:41:00+00:00 2024-02-21 00:01:00+00:00     5.0
16                  redis  log.error_count      5 2024-02-20 23:41:00+00:00 2024-02-21 00:01:00+00:00     0.0, 460, 460, 0, NaT, NaT, 0, Empty DataFrame
Columns: [cmdb_id, message_count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric variable if present; otherwise load it.
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# 1) Parse timestamp with UTC
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Incident window (UTC)
incident_start = pd.to_datetime("2024-02-20 23:34:32", utc=True)
incident_end   = pd.to_datetime("2024-02-21 00:04:32", utc=True)

# Filter metric rows to only candidate services (for scanning groups)
mf = metric[metric['cmdb_id'].isin(candidates)].copy()

# 2) For each (cmdb_id, kpi_name) group compute global P95 and P05 using entire available series (before window filter)
p95 = mf.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value': 'p95'})
p05 = mf.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.05).reset_index().rename(columns={'value': 'p05'})
thresholds = pd.merge(p95, p05, on=['cmdb_id', 'kpi_name'])

# Count total groups scanned
total_groups_scanned = int(thresholds.shape[0])

# 3) Filter metric rows to the incident window
mf_window = mf[(mf['timestamp'] >= incident_start) & (mf['timestamp'] <= incident_end)].copy()

# 4) For each group compute window aggregates
# Basic window aggregates
wnd_agg = mf_window.groupby(['cmdb_id', 'kpi_name']).agg(
    count_in_window=('value', 'size'),
    min_ts_in_window=('timestamp', 'min'),
    max_ts_in_window=('timestamp', 'max'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min')
).reset_index()

# Merge thresholds into windowed data for anomaly detection per-row
mf_window_thresh = mf_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Determine anomalies per row
mf_window_thresh['anomaly_high'] = mf_window_thresh['value'] > mf_window_thresh['p95']
mf_window_thresh['anomaly_low']  = mf_window_thresh['value'] < mf_window_thresh['p05']
mf_window_thresh['any_anomaly']  = mf_window_thresh['anomaly_high'] | mf_window_thresh['anomaly_low']

# Aggregate anomaly counts and earliest anomaly timestamp per group
anom_agg_counts = mf_window_thresh.groupby(['cmdb_id', 'kpi_name']).agg(
    anomaly_high_count=('anomaly_high', 'sum'),
    anomaly_low_count=('anomaly_low', 'sum'),
    earliest_anomaly_timestamp=('timestamp', lambda s: s[mf_window_thresh.loc[s.index, 'any_anomaly']].min() if s.size>0 else pd.NaT)
).reset_index()

# Note: The lambda uses index alignment to pick earliest anomaly timestamp; handle groups with no anomalies:
# If earliest_anomaly_timestamp is NaT, leave as NaT.

# Combine window aggregates, anomaly counts, and thresholds
result = wnd_agg.merge(anom_agg_counts, on=['cmdb_id', 'kpi_name'], how='left').merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Fill NaNs for groups that had window rows but no anomalies
result[['anomaly_high_count', 'anomaly_low_count']] = result[['anomaly_high_count', 'anomaly_low_count']].fillna(0).astype(int)
result['earliest_anomaly_timestamp'] = pd.to_datetime(result['earliest_anomaly_timestamp'], utc=True)

# 5) Filter to groups that have anomalies and prepare compact output sorted by total anomalies desc, limit 20
result['anomaly_total'] = result['anomaly_high_count'] + result['anomaly_low_count']
anomalies_in_window = result[result['anomaly_total'] > 0].copy()
anomalies_in_window = anomalies_in_window.sort_values('anomaly_total', ascending=False).head(20)

# Select and order columns as requested
anomalies_compact = anomalies_in_window[
    [
        'cmdb_id', 'kpi_name', 'count_in_window',
        'anomaly_high_count', 'anomaly_low_count',
        'p95', 'p05',
        'earliest_anomaly_timestamp',
        'min_ts_in_window', 'max_ts_in_window',
        'max_value_in_window', 'min_value_in_window'
    ]
].reset_index(drop=True)

# 6) Summary counts
groups_with_anomalies = int(anomalies_compact.shape[0])

# Display compact outputs
anomalies_compact, total_groups_scanned, groups_with_anomalies

```
Out[2]:
```
```python
# Compose a concise plain-English summary from the previously computed results.
# Uses anomalies_compact, total_groups_scanned, groups_with_anomalies from the kernel.

services = anomalies_compact['cmdb_id'].unique().tolist()
top_kpis = anomalies_compact['kpi_name'].value_counts().head(10).index.tolist()
earliest = anomalies_compact['earliest_anomaly_timestamp'].min()
latest = anomalies_compact['earliest_anomaly_timestamp'].max()

summary = (
    f"Anomaly detection summary (UTC incident window 2024-02-20 23:34:32 to 2024-02-21 00:04:32):\n"
    f"- Scanned {total_groups_scanned} (cmdb_id, kpi_name) groups for the five candidate services.\n"
    f"- {groups_with_anomalies} groups had anomalies inside the incident window (anomaly_high or anomaly_low).\n"
    f"- Affected services: {', '.join(services)}.\n"
    f"- Most-impacted KPIs (top examples): {', '.join(top_kpis)}.\n"
    f"- Typical anomaly pattern: groups in the result had non-zero counts of both high and low anomalies (commonly 2 high and 2 low per group in this window).\n"
    f"- Earliest anomaly observed at {earliest} UTC; latest earliest-anomaly timestamps at {latest} UTC (anomalies occurred mostly starting around 2024-02-20 23:37:00 UTC).\n\n"
    f"Conclusion: Multiple resource and latency KPIs (cpu, mem, workload, latency-50/90) across checkoutservice, currencyservice, emailservice, productcatalogservice, and recommendationservice exhibited both high and low outliers during the incident window. These groups should be prioritized for investigation."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count_in_window  anomaly_high_count  anomaly_low_count           p95           p05 earliest_anomaly_timestamp          min_ts_in_window          max_ts_in_window  max_value_in_window  min_value_in_window
0         checkoutservice         cpu               25                   2                  2  4.842630e-01  2.636119e-01  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         5.468511e-01         1.651386e-01
1         checkoutservice  latency-50               25                   2                  2  1.865165e-01  7.924395e-02  2024-02-20 23:38:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         1.906545e-01         6.731961e-02
2         checkoutservice  latency-90               25                   2                  2  5.603390e-01  2.122642e-01  2024-02-20 23:39:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         6.666806e-01         2.105783e-01
3         checkoutservice         mem               25                   2                  2  1.829337e+07  1.010613e+07  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         1.918157e+07         9.617476e+06
4         checkoutservice    workload               25                   2                  2  1.166067e+00  6.998600e-01  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         1.464400e+00         3.405357e-01
5         currencyservice         cpu               25                   2                  2  3.293933e+01  2.815187e+01  2024-02-20 23:41:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         3.356136e+01         2.776344e+01
6         currencyservice  latency-50               25                   2                  2  5.287978e-03  3.657595e-03  2024-02-20 23:44:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         5.460467e-03         3.632383e-03
7            emailservice         cpu               25                   2                  2  2.981270e-01  2.464652e-01  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         3.152189e-01         2.383237e-01
8         currencyservice  latency-90               25                   2                  2  5.964442e-02  1.345829e-02  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         7.823598e-02         1.317170e-02
9         currencyservice         mem               25                   2                  2  7.431565e+07  4.187178e+07  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         7.865740e+07         4.173449e+07
10        currencyservice    workload               25                   2                  2  8.275727e+01  7.197678e+01  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         8.361897e+01         5.907871e+01
11  productcatalogservice  latency-50               25                   2                  2  5.363557e-03  1.680292e-03  2024-02-20 23:39:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         5.607752e-03         1.653074e-03
12  productcatalogservice         cpu               25                   2                  2  1.986661e+01  3.180968e+00  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         1.992567e+01         2.759939e+00
13           emailservice    workload               25                   2                  2  1.160957e+00  6.938033e-01  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         1.479950e+00         3.904643e-01
14           emailservice         mem               25                   2                  2  8.477523e+07  4.337691e+07  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         8.682346e+07         4.337254e+07
15  recommendationservice    workload               25                   2                  2  1.974135e+01  1.707417e+01  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         2.003555e+01         9.612036e+00
16  recommendationservice         mem               25                   2                  2  8.553027e+07  4.382624e+07  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         8.650547e+07         4.355591e+07
17  recommendationservice  latency-50               25                   2                  2  6.497258e-02  6.979841e-03  2024-02-20 23:39:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         6.623350e-02         6.939078e-03
18  recommendationservice  latency-90               25                   2                  2  9.325841e-02  9.491308e-03  2024-02-20 23:39:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         9.337848e-02         9.460569e-03
19  productcatalogservice         mem               25                   2                  2  1.341279e+08  1.167884e+07  2024-02-20 23:40:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00         1.341724e+08         1.166602e+07, 34, 20)```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing 'trace' variable if present; otherwise load it.
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# 1) Parse timestamp as UTC
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Incident window (UTC)
incident_start = pd.to_datetime("2024-02-20 23:34:32", utc=True)
incident_end   = pd.to_datetime("2024-02-21 00:04:32", utc=True)

# Filter to candidate services (for threshold computation)
tf = trace[trace['cmdb_id'].isin(candidates)].copy()

# 2) Compute global P95 and P05 per (cmdb_id, trace_name) using the full available series
p95 = tf.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})
p05 = tf.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.05).reset_index().rename(columns={'value':'p05'})
thresholds = pd.merge(p95, p05, on=['cmdb_id', 'trace_name'])

# Summary: total groups scanned
total_groups_scanned = int(thresholds.shape[0])

# 3) Filter trace rows to the incident window
tf_window = tf[(tf['timestamp'] >= incident_start) & (tf['timestamp'] <= incident_end)].copy()

# 4) Window aggregates per group
wnd_agg = tf_window.groupby(['cmdb_id', 'trace_name']).agg(
    count_in_window=('value', 'size'),
    min_ts_in_window=('timestamp', 'min'),
    max_ts_in_window=('timestamp', 'max'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min')
).reset_index()

# Merge thresholds into windowed rows for per-row anomaly detection
tf_w_thresh = tf_window.merge(thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Flag anomalies per row
tf_w_thresh['anomaly_high'] = tf_w_thresh['value'] > tf_w_thresh['p95']
tf_w_thresh['anomaly_low']  = tf_w_thresh['value'] < tf_w_thresh['p05']
tf_w_thresh['any_anomaly']  = tf_w_thresh['anomaly_high'] | tf_w_thresh['anomaly_low']

# Aggregate anomaly counts from rows that are inside the window
if not tf_w_thresh.empty:
    anom_rows = tf_w_thresh[tf_w_thresh['any_anomaly']].copy()
    anom_counts = anom_rows.groupby(['cmdb_id', 'trace_name']).agg(
        anomaly_high_count=('anomaly_high', 'sum'),
        anomaly_low_count=('anomaly_low', 'sum'),
    ).reset_index()
    earliest = anom_rows.groupby(['cmdb_id', 'trace_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp'})
    anom_summary = anom_counts.merge(earliest, on=['cmdb_id','trace_name'])
else:
    anom_summary = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_high_count','anomaly_low_count','earliest_anomaly_timestamp'])

# Combine window aggregates, anomaly summary, and thresholds
combined = wnd_agg.merge(anom_summary, on=['cmdb_id', 'trace_name'], how='left').merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for groups with no anomalies
combined['anomaly_high_count'] = combined['anomaly_high_count'].fillna(0).astype(int)
combined['anomaly_low_count']  = combined['anomaly_low_count'].fillna(0).astype(int)
combined['earliest_anomaly_timestamp'] = pd.to_datetime(combined['earliest_anomaly_timestamp'], utc=True)

# 5) Filter to groups with anomalies and prepare compact output sorted by anomaly total desc, limit 20
combined['anomaly_total'] = combined['anomaly_high_count'] + combined['anomaly_low_count']
anomalies_compact = combined[combined['anomaly_total'] > 0].copy()
anomalies_compact = anomalies_compact.sort_values('anomaly_total', ascending=False).head(20)

# Select requested columns and ensure order
anomalies_compact = anomalies_compact[
    [
        'cmdb_id', 'trace_name', 'count_in_window',
        'anomaly_high_count', 'anomaly_low_count',
        'p95', 'p05',
        'earliest_anomaly_timestamp',
        'min_ts_in_window', 'max_ts_in_window',
        'max_value_in_window', 'min_value_in_window'
    ]
].reset_index(drop=True)

# 6) Summary counts
groups_with_anomalies = int(anomalies_compact.shape[0])

# Return compact outputs (DataFrame and two integers)
anomalies_compact, total_groups_scanned, groups_with_anomalies

```
Out[3]:
```
```python
# Compose a concise plain-English summary from the trace anomaly results already in the kernel.
# Uses anomalies_compact, total_groups_scanned, groups_with_anomalies.

import pandas as pd

# Ensure variables exist
try:
    anomalies_compact
    total_groups_scanned
    groups_with_anomalies
except NameError:
    raise RuntimeError("Required variables not found in kernel: run the trace analysis cell first.")

affected_services = sorted(anomalies_compact['cmdb_id'].unique().tolist())
top_traces = anomalies_compact['trace_name'].value_counts().head(6).index.tolist()
earliest = anomalies_compact['earliest_anomaly_timestamp'].min()
latest = anomalies_compact['earliest_anomaly_timestamp'].max()
avg_high = anomalies_compact['anomaly_high_count'].mean()
avg_low = anomalies_compact['anomaly_low_count'].mean()

summary = (
    f"Trace anomaly summary (UTC incident window 2024-02-20 23:34:32 to 2024-02-21 00:04:32):\n"
    f"- Scanned {total_groups_scanned} (cmdb_id, trace_name) groups for the five candidate services.\n"
    f"- {groups_with_anomalies} groups had anomalies inside the window (value > P95 or value < P05).\n"
    f"- Affected services: {', '.join(affected_services)}.\n"
    f"- Most-impacted trace features (examples): {', '.join(top_traces)}.\n"
    f"- Typical anomaly counts for affected groups: average anomaly_high_count ≈ {avg_high:.1f}, "
    f"average anomaly_low_count ≈ {avg_low:.1f} (many groups show 2 high and 2 low anomalies in this window).\n"
    f"- Earliest anomaly observed at {pd.to_datetime(earliest).isoformat()} UTC; latest earliest-anomaly timestamps at {pd.to_datetime(latest).isoformat()} UTC.\n\n"
    f"Conclusion: Multiple trace features (durations and row_count metrics) primarily in checkoutservice and currencyservice showed both high and low outliers during the incident window. These trace groups should be prioritized for further investigation."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                trace_name  count_in_window  anomaly_high_count  anomaly_low_count         p95         p05 earliest_anomaly_timestamp          min_ts_in_window          max_ts_in_window  max_value_in_window  min_value_in_window
0   checkoutservice  trace.from_checkoutservice.duration_mean               25                   2                  2    0.021787    0.004074  2024-02-20 23:43:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.028594             0.003975
1   checkoutservice   trace.from_checkoutservice.duration_p95               25                   2                  2    0.088514    0.007358  2024-02-20 23:46:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.089213             0.007063
2   checkoutservice      trace.from_checkoutservice.row_count               25                   2                  2  302.600000  150.400000  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00           361.000000            67.000000
3   checkoutservice  trace.from_frontendservice.duration_mean               25                   2                  2    0.213623    0.084571  2024-02-20 23:38:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.274363             0.074404
4   checkoutservice   trace.from_frontendservice.duration_p95               25                   2                  2    0.607247    0.137298  2024-02-20 23:40:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.933520             0.132891
5   checkoutservice             trace.from_root.duration_mean               25                   2                  2    0.017628    0.006520  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.018150             0.005167
6   checkoutservice                 trace.from_root.row_count               25                   2                  2  152.200000   75.800000  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00           165.000000            32.000000
7   checkoutservice              trace.from_root.duration_p95               25                   2                  2    0.073830    0.011258  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.081756             0.010134
8   checkoutservice    trace.to_checkoutservice.duration_mean               25                   2                  2    0.021787    0.004074  2024-02-20 23:43:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.028594             0.003975
9   checkoutservice     trace.to_checkoutservice.duration_p95               25                   2                  2    0.088514    0.007358  2024-02-20 23:46:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.089213             0.007063
10  checkoutservice        trace.to_currencyservice.row_count               25                   2                  2  128.200000   60.600000  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00           141.000000            21.000000
11  checkoutservice        trace.to_checkoutservice.row_count               25                   2                  2  302.600000  150.400000  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00           361.000000            67.000000
12  checkoutservice    trace.to_currencyservice.duration_mean               25                   2                  2    0.000845    0.000139  2024-02-20 23:39:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.002218             0.000137
13  checkoutservice     trace.to_currencyservice.duration_p95               25                   2                  2    0.000295    0.000197  2024-02-20 23:44:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000380             0.000196
14  checkoutservice        trace.to_emailservice.duration_p95               25                   2                  2    0.000698    0.000420  2024-02-20 23:38:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000867             0.000352
15  checkoutservice       trace.to_emailservice.duration_mean               25                   2                  2    0.000330    0.000261  2024-02-20 23:38:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000343             0.000254
16  checkoutservice      trace.to_paymentservice.duration_p95               25                   2                  2    0.002193    0.000767  2024-02-20 23:38:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.002358             0.000669
17  checkoutservice     trace.to_paymentservice.duration_mean               25                   2                  2    0.000697    0.000461  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000900             0.000435
18  currencyservice             trace.from_root.duration_mean               25                   2                  2    0.000099    0.000043  2024-02-20 23:44:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000159             0.000043
19  currencyservice              trace.from_root.duration_p95               25                   2                  2    0.000312    0.000056  2024-02-20 23:44:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000651             0.000051, 88, 20)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing 'trace' variable if present; otherwise load it.
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# 1) Ensure timestamps parsed as UTC
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Candidate services (source_cmdb_id)
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Incident window (UTC)
incident_start = pd.to_datetime("2024-02-20 23:34:32", utc=True)
incident_end   = pd.to_datetime("2024-02-21 00:04:32", utc=True)

# Filter trace rows to only candidate sources for threshold calculation and windowed analysis
tf = trace[trace['cmdb_id'].isin(candidates)].copy()

# 2) Compute global P95 and P05 per (cmdb_id, trace_name) using the full available series (before window filter)
thresholds = tf.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    p95=lambda s: s.quantile(0.95),
    p05=lambda s: s.quantile(0.05)
).reset_index()

# 3) Filter trace rows to the incident window
tf_window = tf[(tf['timestamp'] >= incident_start) & (tf['timestamp'] <= incident_end)].copy()

# 4) For each (cmdb_id, trace_name) compute window aggregates and anomaly counts using the precomputed thresholds
# Window aggregates
wnd_agg = tf_window.groupby(['cmdb_id', 'trace_name']).agg(
    count_in_window=('value', 'size'),
    min_ts_in_window=('timestamp', 'min'),
    max_ts_in_window=('timestamp', 'max'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min')
).reset_index()

# Merge thresholds into windowed rows for per-row anomaly detection
tf_w_thresh = tf_window.merge(thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Flag anomalies per row
tf_w_thresh['anomaly_high'] = tf_w_thresh['value'] > tf_w_thresh['p95']
tf_w_thresh['anomaly_low']  = tf_w_thresh['value'] < tf_w_thresh['p05']
tf_w_thresh['any_anomaly']  = tf_w_thresh['anomaly_high'] | tf_w_thresh['anomaly_low']

# Per-group anomaly counts and earliest anomaly timestamp
if not tf_w_thresh.empty:
    anom_rows = tf_w_thresh[tf_w_thresh['any_anomaly']].copy()
    anom_counts = anom_rows.groupby(['cmdb_id', 'trace_name']).agg(
        anomaly_high_count=('anomaly_high', 'sum'),
        anomaly_low_count=('anomaly_low', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min')
    ).reset_index()
else:
    anom_counts = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_high_count','anomaly_low_count','earliest_anomaly_timestamp'])

# Combine window aggregates with anomaly counts and thresholds to form per-(cmdb_id, trace_name) groups
combined = wnd_agg.merge(anom_counts, on=['cmdb_id','trace_name'], how='left').merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for groups without anomalies
combined['anomaly_high_count'] = combined['anomaly_high_count'].fillna(0).astype(int)
combined['anomaly_low_count']  = combined['anomaly_low_count'].fillna(0).astype(int)
combined['earliest_anomaly_timestamp'] = pd.to_datetime(combined['earliest_anomaly_timestamp'], utc=True)

# 1) Parse trace_name to extract direction token and target service (or 'root')
def extract_target(trace_name):
    # split by '.' and take the second segment if exists (e.g., 'trace.to_currencyservice.duration_mean' -> 'to_currencyservice')
    parts = str(trace_name).split('.')
    token = parts[1] if len(parts) > 1 else ''
    if token.startswith('to_'):
        return token[3:]
    if token.startswith('from_'):
        # for 'from_root' -> return 'root'; for 'from_frontendservice' -> 'frontendservice'
        return token[5:]
    # fallback
    return 'root'

combined['target_service'] = combined['trace_name'].apply(extract_target)
combined['source_cmdb_id'] = combined['cmdb_id']

# 2) For each pair (source_cmdb_id, target_service) aggregate anomaly severity and deviations
# Compute per-group max_deviation: high -> max_value - p95 ; low -> p05 - min_value ; take larger
# Fill missing p95/p05 with NaN-safe values
combined['p95'] = combined['p95'].astype(float)
combined['p05'] = combined['p05'].astype(float)

# Compute deviations (use 0 if no corresponding anomaly count)
combined['high_dev'] = np.where(combined['anomaly_high_count'] > 0, combined['max_value_in_window'] - combined['p95'], 0.0)
combined['low_dev']  = np.where(combined['anomaly_low_count'] > 0, combined['p05'] - combined['min_value_in_window'], 0.0)
combined['group_max_deviation'] = combined[['high_dev', 'low_dev']].max(axis=1)

# Sum anomalies per pair and aggregate earliest anomaly timestamp and other window stats
pair_agg = combined.groupby(['source_cmdb_id', 'target_service']).agg(
    total_anomaly_count=('anomaly_high_count', 'sum'),  # will add low later
    total_anomaly_count_low=('anomaly_low_count', 'sum'),
    earliest_anomaly_timestamp=('earliest_anomaly_timestamp', 'min'),
    max_deviation=('group_max_deviation', 'max'),
    min_ts_in_window=('min_ts_in_window', 'min'),
    max_ts_in_window=('max_ts_in_window', 'max'),
    max_value_in_window=('max_value_in_window', 'max'),
    min_value_in_window=('min_value_in_window', 'min')
).reset_index()

# combine high and low counts into single total
pair_agg['total_anomaly_count'] = pair_agg['total_anomaly_count'] + pair_agg['total_anomaly_count_low']
pair_agg = pair_agg.drop(columns=['total_anomaly_count_low'])

# 3) Filter to pairs that have anomalies in-window and prepare compact table
pair_agg_compact = pair_agg[pair_agg['total_anomaly_count'] > 0].copy()
pair_agg_compact = pair_agg_compact.sort_values('total_anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Rename columns per requested output order
pair_agg_compact = pair_agg_compact[
    [
        'source_cmdb_id', 'target_service', 'total_anomaly_count', 'earliest_anomaly_timestamp',
        'max_deviation', 'min_ts_in_window', 'max_ts_in_window', 'max_value_in_window', 'min_value_in_window'
    ]
]

# 4) Summary counts
total_pairs_scanned = int(combined[['source_cmdb_id','target_service']].drop_duplicates().shape[0])
pairs_with_anomalies = int(pair_agg_compact.shape[0])

# Keep full intermediate tables in kernel variables for follow-ups:
# - combined (per cmdb_id, trace_name groups with thresholds and anomalies)
# - pair_agg (aggregated per source->target before filtering)
# - pair_agg_compact (final compact table to return)

pair_agg_compact, total_pairs_scanned, pairs_with_anomalies

```
Out[4]:
```
```python
# Compose a concise plain-English summary from the source->target pair anomaly results in the kernel.
# Uses pair_agg_compact, total_pairs_scanned, pairs_with_anomalies variables produced earlier.

try:
    pair_agg_compact
    total_pairs_scanned
    pairs_with_anomalies
except NameError:
    raise RuntimeError("Required variables not found in kernel: run the source->target trace mapping cell first.")

# Top 5 pairs for quick highlights
top5 = pair_agg_compact.head(5)[
    ['source_cmdb_id','target_service','total_anomaly_count','earliest_anomaly_timestamp','max_deviation','max_value_in_window','min_value_in_window']
]

earliest_overall = pair_agg_compact['earliest_anomaly_timestamp'].min()

lines = [
    f"Trace-direction anomaly mapping (incident window 2024-02-20 23:34:32 to 2024-02-21 00:04:32 UTC):",
    f"- Scanned {total_pairs_scanned} unique source->target pairs for the five candidate services.",
    f"- Pairs with anomalies in-window: {pairs_with_anomalies}.",
    f"- Earliest anomaly observed at {pd.to_datetime(earliest_overall).isoformat()} UTC.",
    "",
    "Top 5 source->target pairs by total anomaly count and key metrics:"
]

for _, r in top5.iterrows():
    lines.append(
        f"  * {r.source_cmdb_id} -> {r.target_service}: total_anomaly_count={int(r.total_anomaly_count)}, "
        f"earliest={pd.to_datetime(r.earliest_anomaly_timestamp).isoformat()}, "
        f"max_deviation={float(r.max_deviation):.1f}, max_value={float(r.max_value_in_window):.1f}, min_value={float(r.min_value_in_window):.1f}"
    )

lines.append("")
lines.append("Notable observations:")
lines.append("- The largest anomaly counts are self-directed for checkoutservice and recommendationservice (each with 24 total anomalies).")
lines.append("- Several cross-service paths show elevated anomalies (e.g., currencyservice <-> checkoutservice and recommendationservice -> productcatalogservice).")
lines.append("- Some pairs show very large deviations in absolute values (e.g., productcatalogservice -> frontendservice with max_value ~4044 and large max_deviation), indicating substantial outliers in row_count/throughput-type traces.")
lines.append("")
lines.append("Recommendation: prioritize investigation of checkoutservice and recommendationservice self-flows and the high-deviation pairs (productcatalogservice->frontendservice, currencyservice->frontendservice) as they exhibit the largest anomaly counts and deviations.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(           source_cmdb_id         target_service  total_anomaly_count earliest_anomaly_timestamp  max_deviation          min_ts_in_window          max_ts_in_window  max_value_in_window  min_value_in_window
0         checkoutservice        checkoutservice                   24  2024-02-20 23:37:00+00:00           83.4 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                361.0                  0.0
1   recommendationservice  recommendationservice                   24  2024-02-20 23:37:00+00:00          174.4 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                614.0                  0.0
2         currencyservice        checkoutservice                   12  2024-02-20 23:37:00+00:00           39.6 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                141.0                  0.0
3         checkoutservice        currencyservice                   12  2024-02-20 23:37:00+00:00           39.6 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                141.0                  0.0
4   recommendationservice  productcatalogservice                   12  2024-02-20 23:37:00+00:00          173.6 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                614.0                  0.0
5   recommendationservice        frontendservice                   12  2024-02-20 23:37:00+00:00          174.4 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                614.0                  0.0
6         currencyservice        frontendservice                   12  2024-02-20 23:37:00+00:00          456.8 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00               2462.0                  0.0
7         checkoutservice                   root                   12  2024-02-20 23:37:00+00:00           43.8 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                165.0                  0.0
8   productcatalogservice  recommendationservice                   12  2024-02-20 23:37:00+00:00          173.6 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                614.0                  0.0
9         checkoutservice           emailservice                   11  2024-02-20 23:37:00+00:00            9.0 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 44.0                  0.0
10        checkoutservice        frontendservice                   11  2024-02-20 23:37:00+00:00            9.0 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 44.0                  0.0
11        checkoutservice  productcatalogservice                   11  2024-02-20 23:37:00+00:00           30.0 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 97.0                  0.0
12  productcatalogservice        checkoutservice                   11  2024-02-20 23:37:00+00:00           30.0 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 97.0                  0.0
13  productcatalogservice        frontendservice                   11  2024-02-20 23:37:00+00:00         1174.0 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00               4044.0                  0.0
14           emailservice        checkoutservice                   11  2024-02-20 23:37:00+00:00            9.0 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 44.0                  0.0
15        checkoutservice         paymentservice                   11  2024-02-20 23:37:00+00:00            9.0 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 44.0                  0.0
16           emailservice                   root                   10  2024-02-20 23:37:00+00:00            2.4 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 24.0                  0.0
17        currencyservice                   root                   10  2024-02-20 23:37:00+00:00            1.2 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 12.0                  0.0
18  productcatalogservice                   root                   10  2024-02-20 23:37:00+00:00            2.4 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 24.0                  0.0
19  recommendationservice                   root                   10  2024-02-20 23:37:00+00:00            2.4 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00                 24.0                  0.0, 20, 20)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing kernel variables when available
try:
    pair_agg_compact
except NameError:
    raise RuntimeError("pair_agg_compact not found in kernel. Run the trace source->target aggregation step first.")

# Candidates
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute trace-side aggregates per candidate (target_service)
# Ensure earliest_anomaly_timestamp is datetime
if 'earliest_anomaly_timestamp' in pair_agg_compact.columns:
    pair_agg_compact['earliest_anomaly_timestamp'] = pd.to_datetime(pair_agg_compact['earliest_anomaly_timestamp'], utc=True)

trace_stats = (
    pair_agg_compact.groupby('target_service', dropna=True)
    .agg(
        trace_target_anomaly_count=('total_anomaly_count', 'sum'),
        trace_earliest=('earliest_anomaly_timestamp', 'min'),
        trace_max_deviation=('max_deviation', 'max')
    )
    .reset_index()
)

# Keep only candidate targets (treat 'root' as not matching any candidate)
trace_stats = trace_stats[trace_stats['target_service'].isin(candidates)].set_index('target_service')

# 2) Prepare metric-side anomalies: try to reuse anomalies_compact if it contains metric groups (kpi_name),
# otherwise recompute from metric DataFrame stored in kernel.
use_existing_metric_anomalies = False
if 'anomalies_compact' in globals() and isinstance(anomalies_compact, pd.DataFrame) and 'kpi_name' in anomalies_compact.columns:
    metric_groups_df = anomalies_compact.copy()
    use_existing_metric_anomalies = True
else:
    # Recompute metric-group anomalies for candidate services
    try:
        metric
    except NameError:
        raise RuntimeError("Metric dataframe 'metric' not found in kernel; cannot compute metric anomalies.")
    # Ensure timestamp parsed
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
    # Filter to candidates
    mf = metric[metric['cmdb_id'].isin(candidates)].copy()
    # Compute global thresholds per (cmdb_id, kpi_name)
    thresholds = mf.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        p95=lambda s: s.quantile(0.95),
        p05=lambda s: s.quantile(0.05)
    ).reset_index()
    # Incident window
    incident_start = pd.to_datetime("2024-02-20 23:34:32", utc=True)
    incident_end   = pd.to_datetime("2024-02-21 00:04:32", utc=True)
    mf_window = mf[(mf['timestamp'] >= incident_start) & (mf['timestamp'] <= incident_end)].copy()
    if mf_window.empty:
        metric_groups_df = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','count_in_window','anomaly_high_count','anomaly_low_count',
            'p95','p05','earliest_anomaly_timestamp','min_ts_in_window','max_ts_in_window',
            'max_value_in_window','min_value_in_window'
        ])
    else:
        # Merge thresholds into window rows
        mf_w = mf_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
        mf_w['anomaly_high'] = mf_w['value'] > mf_w['p95']
        mf_w['anomaly_low']  = mf_w['value'] < mf_w['p05']
        mf_w['any_anomaly']  = mf_w['anomaly_high'] | mf_w['anomaly_low']
        # Per-group window aggregates
        wnd_agg = mf_w.groupby(['cmdb_id','kpi_name']).agg(
            count_in_window=('value','size'),
            min_ts_in_window=('timestamp','min'),
            max_ts_in_window=('timestamp','max'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
            p95=('p95','first'),
            p05=('p05','first')
        ).reset_index()
        # Anomaly counts and earliest timestamp
        anom = mf_w[mf_w['any_anomaly']].groupby(['cmdb_id','kpi_name']).agg(
            anomaly_high_count=('anomaly_high','sum'),
            anomaly_low_count=('anomaly_low','sum'),
            earliest_anomaly_timestamp=('timestamp','min')
        ).reset_index()
        metric_groups_df = wnd_agg.merge(anom, on=['cmdb_id','kpi_name'], how='left')
        metric_groups_df['anomaly_high_count'] = metric_groups_df['anomaly_high_count'].fillna(0).astype(int)
        metric_groups_df['anomaly_low_count']  = metric_groups_df['anomaly_low_count'].fillna(0).astype(int)
        metric_groups_df['earliest_anomaly_timestamp'] = pd.to_datetime(metric_groups_df['earliest_anomaly_timestamp'], utc=True)

# 3) From metric_groups_df compute per-service metrics
# Compute per-group deviation
if not metric_groups_df.empty:
    metric_groups_df['p95'] = metric_groups_df['p95'].astype(float)
    metric_groups_df['p05'] = metric_groups_df['p05'].astype(float)
    metric_groups_df['high_dev'] = np.where(metric_groups_df['anomaly_high_count'] > 0,
                                            metric_groups_df['max_value_in_window'] - metric_groups_df['p95'],
                                            0.0)
    metric_groups_df['low_dev']  = np.where(metric_groups_df['anomaly_low_count'] > 0,
                                            metric_groups_df['p05'] - metric_groups_df['min_value_in_window'],
                                            0.0)
    metric_groups_df['group_max_deviation'] = metric_groups_df[['high_dev','low_dev']].max(axis=1)
else:
    metric_groups_df['group_max_deviation'] = pd.Series(dtype=float)

# Aggregate per service
metric_stats = (
    metric_groups_df.assign(service=metric_groups_df['cmdb_id'])
    .groupby('service', dropna=True)
    .agg(
        metric_anomaly_groups=('kpi_name', lambda s: int(s.size if s.size>0 else 0)),  # counts of groups present (will filter next)
        metric_max_deviation=('group_max_deviation', 'max')
    )
    .reset_index()
)

# metric_anomaly_groups should count only groups with any anomaly
if not metric_groups_df.empty:
    metric_groups_with_anom = metric_groups_df[(metric_groups_df['anomaly_high_count'] > 0) | (metric_groups_df['anomaly_low_count'] > 0)]
    metric_stats = (
        metric_groups_with_anom.groupby('cmdb_id', dropna=True)
        .agg(
            metric_anomaly_groups=('kpi_name', 'nunique'),
            metric_max_deviation=('group_max_deviation', 'max')
        )
        .reset_index()
        .rename(columns={'cmdb_id':'service'})
    )
else:
    metric_stats = pd.DataFrame(columns=['service','metric_anomaly_groups','metric_max_deviation'])

metric_stats = metric_stats.set_index('service')

# 4) Build final summary DataFrame for the five candidates
rows = []
for svc in candidates:
    trace_row = trace_stats.loc[svc] if svc in trace_stats.index else pd.Series({
        'trace_target_anomaly_count': 0,
        'trace_earliest': pd.NaT,
        'trace_max_deviation': 0.0
    })
    metric_row = metric_stats.loc[svc] if svc in metric_stats.index else pd.Series({
        'metric_anomaly_groups': 0,
        'metric_max_deviation': 0.0
    })
    rows.append({
        'service': svc,
        'trace_target_anomaly_count': int(trace_row.get('trace_target_anomaly_count', 0)),
        'trace_earliest': pd.to_datetime(trace_row.get('trace_earliest')) if pd.notna(trace_row.get('trace_earliest')) else pd.NaT,
        'trace_max_deviation': float(trace_row.get('trace_max_deviation', 0.0) or 0.0),
        'metric_anomaly_groups': int(metric_row.get('metric_anomaly_groups', 0) or 0),
        'metric_max_deviation': float(metric_row.get('metric_max_deviation', 0.0) or 0.0)
    })

final_df = pd.DataFrame(rows)

# 5) Sort and limit to 5 rows
final_df = final_df.sort_values(['trace_target_anomaly_count', 'metric_max_deviation'], ascending=[False, False]).reset_index(drop=True).head(5)

# Summary numbers
total_services = len(candidates)
services_with_trace_anomalies = int((final_df['trace_target_anomaly_count'] > 0).sum())

# Display compact outputs
final_df, total_services, services_with_trace_anomalies

```
Out[5]:
```
```python
# Compose a concise plain-English summary string from existing kernel variables:
# final_df, total_services, services_with_trace_anomalies.

svc_df = final_df.copy()
svc_df['trace_earliest'] = svc_df['trace_earliest'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
lines = [
    f"Per-candidate-service summary for the incident window (2024-02-20 23:34:32 to 2024-02-21 00:04:32 UTC):",
    f"- Services evaluated: {total_services}. Services with trace-target anomalies: {services_with_trace_anomalies}.",
    ""
]
for _, r in svc_df.iterrows():
    lines.append(
        f"- {r.service}: trace_target_anomaly_count={int(r.trace_target_anomaly_count)}, "
        f"trace_earliest={r.trace_earliest}, trace_max_deviation={float(r.trace_max_deviation):.1f}, "
        f"metric_anomaly_groups={int(r.metric_anomaly_groups)}, metric_max_deviation={float(r.metric_max_deviation):.1f}"
    )

lines.append("")
lines.append("Key takeaways:")
lines.append("- checkoutservice and recommendationservice show the highest trace-target anomaly counts and substantial metric deviations; productcatalogservice and currencyservice also show notable metric deviations.")
lines.append("- Earliest anomalies for targets occurred around 2024-02-20 23:37:00 UTC for all affected services.")
lines.append("- Recommendation: prioritize investigation of checkoutservice, recommendationservice, and productcatalogservice (self-flows and cross-service paths) because they combine high trace anomaly counts with large metric deviations.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 service  trace_target_anomaly_count            trace_earliest  trace_max_deviation  metric_anomaly_groups  metric_max_deviation
0        checkoutservice                          58 2024-02-20 23:37:00+00:00                 83.4                      6          8.881981e+05
1  recommendationservice                          36 2024-02-20 23:37:00+00:00                174.4                      6          9.752049e+05
2  productcatalogservice                          23 2024-02-20 23:37:00+00:00                173.6                      7          2.069823e+07
3        currencyservice                          12 2024-02-20 23:37:00+00:00                 39.6                      6          4.341754e+06
4           emailservice                          11 2024-02-20 23:37:00+00:00                  9.0                      6          2.048226e+06, 5, 5)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Recompute metric anomaly groups for the three services because 'anomalies_compact' may have been overwritten.
# Reuse existing 'metric' DataFrame from kernel.
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# 1) Ensure timestamp parsed as UTC
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Target services
services = ['checkoutservice', 'recommendationservice', 'productcatalogservice']

# Incident window (UTC)
incident_start = pd.to_datetime("2024-02-20 23:34:32", utc=True)
incident_end   = pd.to_datetime("2024-02-21 00:04:32", utc=True)

# Filter metric series to the three services for threshold computation
mf = metric[metric['cmdb_id'].isin(services)].copy()

# 2) Compute global P95 and P05 per (cmdb_id, kpi_name) using the full available series
thresholds = mf.groupby(['cmdb_id','kpi_name'])['value'].agg(
    p95=lambda s: s.quantile(0.95),
    p05=lambda s: s.quantile(0.05)
).reset_index()

# 3) Filter metric rows to the incident window
mf_window = mf[(mf['timestamp'] >= incident_start) & (mf['timestamp'] <= incident_end)].copy()

# 4) Compute window aggregates per group and anomaly counts using precomputed thresholds
if mf_window.empty:
    # Prepare empty results
    result_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count_in_window','p95','p05',
        'max_value_in_window','min_value_in_window','deviation','earliest_anomaly_timestamp'
    ])
    groups_scanned = int(thresholds.shape[0])
    groups_with_deviation = 0
else:
    # Merge thresholds into window rows
    mf_w = mf_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    # Window aggregates
    wnd_agg = mf_w.groupby(['cmdb_id','kpi_name']).agg(
        count_in_window=('value','size'),
        min_ts_in_window=('timestamp','min'),
        max_ts_in_window=('timestamp','max'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        p95=('p95','first'),
        p05=('p05','first')
    ).reset_index()
    # Flag anomalies per row
    mf_w['anomaly_high'] = mf_w['value'] > mf_w['p95']
    mf_w['anomaly_low']  = mf_w['value'] < mf_w['p05']
    mf_w['any_anomaly']  = mf_w['anomaly_high'] | mf_w['anomaly_low']
    # Earliest anomaly timestamp per group (if any)
    earliest = mf_w[mf_w['any_anomaly']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp'})
    # Combine
    combined = wnd_agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
    # 2) Compute numeric deviation per group
    combined['p95'] = combined['p95'].astype(float)
    combined['p05'] = combined['p05'].astype(float)
    combined['max_value_in_window'] = combined['max_value_in_window'].astype(float)
    combined['min_value_in_window'] = combined['min_value_in_window'].astype(float)
    combined['dev_high'] = (combined['max_value_in_window'] - combined['p95']).clip(lower=0.0)
    combined['dev_low']  = (combined['p05'] - combined['min_value_in_window']).clip(lower=0.0)
    combined['deviation'] = combined[['dev_high','dev_low']].max(axis=1)
    # 3) Prepare compact table sorted by deviation desc, limit 20
    result_df = combined[
        [
            'cmdb_id','kpi_name','count_in_window','p95','p05',
            'max_value_in_window','min_value_in_window','deviation','earliest_anomaly_timestamp'
        ]
    ].sort_values('deviation', ascending=False).head(20).reset_index(drop=True)
    # 4) Summary numbers
    groups_scanned = int(thresholds.shape[0])  # number of unique (cmdb_id,kpi_name) groups scanned for these services
    groups_with_deviation = int((combined['deviation'] > 0).sum())

# Return compact outputs
result_df, groups_scanned, groups_with_deviation

```
Out[6]:
```
```python
# Compose a plain-English summary string from the metric deviation results in the kernel.
# Uses result_df, groups_scanned, groups_with_deviation produced earlier.

try:
    result_df
    groups_scanned
    groups_with_deviation
except NameError:
    raise RuntimeError("Required variables not found in kernel: run the metric deviation computation first.")

def fmt(x):
    if pd.isna(x):
        return "null"
    if abs(x) >= 1000:
        return f"{x:,.0f}"
    if abs(x) >= 1:
        return f"{x:,.3f}"
    return f"{x:.6f}"

top_rows = result_df.head(3)
lines = [
    "Metric anomaly deviation summary for checkoutservice, recommendationservice, productcatalogservice (incident window):",
    f"- Groups scanned: {int(groups_scanned)}. Groups with deviation > 0: {int(groups_with_deviation)}.",
    "",
    "Top deviations (by group):"
]

for i, row in top_rows.iterrows():
    lines.append(
        f"  {i+1}. {row['cmdb_id']} / {row['kpi_name']}: deviation = {fmt(row['deviation'])}, "
        f"p95 = {fmt(row['p95'])}, p05 = {fmt(row['p05'])}, max = {fmt(row['max_value_in_window'])}, "
        f"min = {fmt(row['min_value_in_window'])}, earliest_anomaly = {pd.to_datetime(row['earliest_anomaly_timestamp'])}"
    )

lines += [
    "",
    "Key observations:",
    "- The largest deviation is productcatalogservice.diskio (deviation ≈ 20,698,230) with earliest anomaly at 2024-02-20 23:51:00 UTC.",
    "- recommendationservice.mem and checkoutservice.mem also show very large deviations (≈975,205 and ≈888,198 respectively), earliest anomalies around 2024-02-20 23:37:00 UTC.",
    "- Overall 19 of 20 scanned (cmdb_id,kpi_name) groups show non-zero deviation, indicating widespread metric outliers across these three services.",
    "",
    "Recommendation: prioritize investigation of productcatalogservice.diskio and the mem KPIs for recommendationservice and checkoutservice."
]

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count_in_window           p95           p05  max_value_in_window  min_value_in_window     deviation earliest_anomaly_timestamp
0   productcatalogservice      diskio               25  8.459010e+09  0.000000e+00         8.479708e+09         0.000000e+00  2.069823e+07  2024-02-20 23:51:00+00:00
1   recommendationservice         mem               25  8.553027e+07  4.382624e+07         8.650547e+07         4.355591e+07  9.752049e+05  2024-02-20 23:37:00+00:00
2         checkoutservice         mem               25  1.829337e+07  1.010613e+07         1.918157e+07         9.617476e+06  8.881981e+05  2024-02-20 23:37:00+00:00
3   productcatalogservice         mem               25  1.341279e+08  1.167884e+07         1.341724e+08         1.166602e+07  4.448256e+04  2024-02-20 23:40:00+00:00
4   productcatalogservice    workload               25  1.500975e+02  1.282294e+02         1.507409e+02         7.180932e+01  5.642010e+01  2024-02-20 23:37:00+00:00
5   recommendationservice    workload               25  1.974135e+01  1.707417e+01         2.003555e+01         9.612036e+00  7.462134e+00  2024-02-20 23:37:00+00:00
6         checkoutservice      socket               25  1.749000e+01  9.000000e+00         1.800000e+01         9.000000e+00  5.100000e-01  2024-02-20 23:37:00+00:00
7   recommendationservice         cpu               25  2.658348e+00  2.131557e+00         2.670983e+00         1.689178e+00  4.423790e-01  2024-02-20 23:37:00+00:00
8   productcatalogservice         cpu               25  1.986661e+01  3.180968e+00         1.992567e+01         2.759939e+00  4.210292e-01  2024-02-20 23:37:00+00:00
9         checkoutservice    workload               25  1.166067e+00  6.998600e-01         1.464400e+00         3.405357e-01  3.593243e-01  2024-02-20 23:37:00+00:00
10  productcatalogservice      socket               25  8.650000e+00  4.000000e+00         9.000000e+00         4.000000e+00  3.500000e-01  2024-02-20 23:37:00+00:00
11        checkoutservice  latency-90               25  5.603390e-01  2.122642e-01         6.666806e-01         2.105783e-01  1.063415e-01  2024-02-20 23:39:00+00:00
12        checkoutservice         cpu               25  4.842630e-01  2.636119e-01         5.468511e-01         1.651386e-01  9.847322e-02  2024-02-20 23:37:00+00:00
13  recommendationservice      socket               25  7.933333e+00  4.000000e+00         8.000000e+00         4.000000e+00  6.666667e-02  2024-02-20 23:37:00+00:00
14        checkoutservice  latency-50               25  1.865165e-01  7.924395e-02         1.906545e-01         6.731961e-02  1.192435e-02  2024-02-20 23:38:00+00:00
15  recommendationservice  latency-50               25  6.497258e-02  6.979841e-03         6.623350e-02         6.939078e-03  1.260915e-03  2024-02-20 23:39:00+00:00
16  productcatalogservice  latency-50               25  5.363557e-03  1.680292e-03         5.607752e-03         1.653074e-03  2.441952e-04  2024-02-20 23:39:00+00:00
17  recommendationservice  latency-90               25  9.325841e-02  9.491308e-03         9.337848e-02         9.460569e-03  1.200693e-04  2024-02-20 23:39:00+00:00
18  productcatalogservice  latency-90               25  8.743029e-02  4.354818e-03         8.749701e-02         4.348957e-03  6.671265e-05  2024-02-20 23:39:00+00:00
19  recommendationservice      diskio               25  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00  0.000000e+00                        NaT, 20, 19)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse existing 'trace' dataframe if present; otherwise load it
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# 1) Ensure timestamp parsed as UTC
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Incident window (UTC)
incident_start = pd.to_datetime("2024-02-20 23:34:32", utc=True)
incident_end   = pd.to_datetime("2024-02-21 00:04:32", utc=True)

# 2) Identify groups where trace_name contains 'productcatalogservice'
mask_pc = trace['trace_name'].str.contains('productcatalogservice', na=False)
trace_pc_all = trace[mask_pc].copy()

# Compute global P95 and P05 per (cmdb_id, trace_name) using the full available series for these groups
thresholds_pc = (
    trace_pc_all
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda s: s.quantile(0.95), p05=lambda s: s.quantile(0.05))
    .reset_index()
)

total_groups_scanned = int(thresholds_pc.shape[0])

# 3) Filter rows to the incident window
trace_pc_window = trace_pc_all[(trace_pc_all['timestamp'] >= incident_start) & (trace_pc_all['timestamp'] <= incident_end)].copy()

# 4) If windowed data is non-empty, compute per-group window aggregates and anomalies using precomputed thresholds
if not trace_pc_window.empty:
    # Window aggregates
    wnd = trace_pc_window.groupby(['cmdb_id', 'trace_name']).agg(
        count_in_window=('value', 'size'),
        min_ts_in_window=('timestamp', 'min'),
        max_ts_in_window=('timestamp', 'max'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min')
    ).reset_index()

    # Merge thresholds for anomaly detection
    w_thresh = trace_pc_window.merge(thresholds_pc, on=['cmdb_id', 'trace_name'], how='left')

    # Flag anomalies per row
    w_thresh['anomaly_high'] = w_thresh['value'] > w_thresh['p95']
    w_thresh['anomaly_low']  = w_thresh['value'] < w_thresh['p05']
    w_thresh['any_anomaly']  = w_thresh['anomaly_high'] | w_thresh['anomaly_low']

    # Aggregate anomaly counts and earliest anomaly timestamp per group
    if w_thresh['any_anomaly'].any():
        anom_rows = w_thresh[w_thresh['any_anomaly']].copy()
        anom_counts = anom_rows.groupby(['cmdb_id', 'trace_name']).agg(
            anomaly_high_count=('anomaly_high', 'sum'),
            anomaly_low_count=('anomaly_low', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        ).reset_index()
    else:
        anom_counts = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_high_count','anomaly_low_count','earliest_anomaly_timestamp'])

    # Combine aggregates, anomalies, and thresholds
    combined = (
        wnd
        .merge(anom_counts, on=['cmdb_id', 'trace_name'], how='left')
        .merge(thresholds_pc, on=['cmdb_id', 'trace_name'], how='left')
    )

    # Fill NaNs for groups without anomalies
    combined['anomaly_high_count'] = combined['anomaly_high_count'].fillna(0).astype(int)
    combined['anomaly_low_count']  = combined['anomaly_low_count'].fillna(0).astype(int)
    combined['earliest_anomaly_timestamp'] = pd.to_datetime(combined['earliest_anomaly_timestamp'], utc=True)

else:
    # Prepare empty combined structure with expected columns
    combined = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count_in_window','min_ts_in_window','max_ts_in_window',
        'max_value_in_window','min_value_in_window','anomaly_high_count','anomaly_low_count',
        'earliest_anomaly_timestamp','p95','p05'
    ])

# 5) Add role column based on trace_name prefix
def classify_role(tn):
    if str(tn).startswith('trace.to_'):
        return 'to_target'
    if str(tn).startswith('trace.from_'):
        return 'from_source'
    return 'other'

combined['role'] = combined['trace_name'].apply(classify_role)

# 6) Prepare compact table: filter to groups with any anomalies and sort by total anomalies desc, limit 20
combined['anomaly_total'] = combined['anomaly_high_count'] + combined['anomaly_low_count']
pc_groups_compact = (
    combined[combined['anomaly_total'] > 0]
    .sort_values('anomaly_total', ascending=False)
    .head(20)
    [
        [
            'cmdb_id','trace_name','role','count_in_window',
            'anomaly_high_count','anomaly_low_count','p95','p05',
            'earliest_anomaly_timestamp','min_ts_in_window','max_ts_in_window',
            'max_value_in_window','min_value_in_window'
        ]
    ]
).reset_index(drop=True)

groups_with_anomalies = int(pc_groups_compact.shape[0])

# Return compact table and small summaries
pc_groups_compact, total_groups_scanned, groups_with_anomalies

```
Out[7]:
```
Summary of traces mentioning "productcatalogservice" (incident window 2024-02-20 23:34:32 — 2024-02-21 00:04:32 UTC):

- Scope:
  - Total distinct (cmdb_id, trace_name) groups scanned that contain "productcatalogservice": 16.
  - Groups with anomalies inside the incident window: 12.

- Roles:
  - Most groups are "to_target" (trace names starting with trace.to_), i.e., these traces show calls directed to productcatalogservice. No significant "from_" source groups were returned in the top anomalous set — productcatalogservice appears primarily as a target in the anomalous traces.

- Timing:
  - Earliest anomalies appear around 2024-02-20 23:37:00 UTC for many groups; some groups show earliest anomalies later (e.g., checkoutservice -> productcatalogservice duration_mean at ~23:50:00 UTC).

- Key anomalous trace groups (highest anomaly counts / notable deviations):
  - frontendservice -> productcatalogservice (row_count): count_in_window=25, anomalies high+low, max_value_in_window=4044, min_value_in_window=1128 — very large throughput/row_count outliers (earliest anomaly 23:37 UTC).
  - recommendationservice -> productcatalogservice (row_count): max_value_in_window=614, min=180 (earliest anomaly 23:37 UTC).
  - checkoutservice -> productcatalogservice (row_count): max_value_in_window=97, min=12 (earliest anomaly 23:37 UTC).
  - Multiple duration-related traces to productcatalogservice (from checkoutservice, frontendservice, recommendationservice, root) show both high and low outliers in duration metrics (earliest anomalies mostly around 23:37–23:40 UTC).
  - A checkoutservice -> productcatalogservice.duration_mean anomaly appears later at ~23:50 UTC.

- Interpretation and recommended next steps:
  - The anomalous traces indicate productcatalogservice was the target of elevated and highly variable request/row counts and duration outliers during the incident window — especially traffic coming from frontendservice, recommendationservice, and checkoutservice.
  - Prioritize investigating incoming load and processing capacity on productcatalogservice (check request rates, queue/backpressure, and resource KPIs), and inspect frontendservice->productcatalogservice and recommendationservice->productcatalogservice call patterns to identify whether increased/irregular traffic caused the anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                    trace_name       role  count_in_window  anomaly_high_count  anomaly_low_count          p95          p05 earliest_anomaly_timestamp          min_ts_in_window          max_ts_in_window  max_value_in_window  min_value_in_window
0         checkoutservice  trace.to_productcatalogservice.duration_mean  to_target               25                   2                  2     0.000023     0.000013  2024-02-20 23:50:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.001364             0.000013
1         checkoutservice      trace.to_productcatalogservice.row_count  to_target               25                   2                  2    93.400000    42.000000  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00            97.000000            12.000000
2         frontendservice  trace.to_productcatalogservice.duration_mean  to_target               25                   2                  2     0.000279     0.000014  2024-02-20 23:38:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000379             0.000014
3         frontendservice      trace.to_productcatalogservice.row_count  to_target               25                   2                  2  3767.600000  2302.000000  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00          4044.000000          1128.000000
4                    root   trace.to_productcatalogservice.duration_p95  to_target               25                   2                  2     0.178491     0.008106  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.182183             0.007537
5   recommendationservice  trace.to_productcatalogservice.duration_mean  to_target               25                   2                  2     0.000194     0.000015  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000228             0.000015
6   recommendationservice   trace.to_productcatalogservice.duration_p95  to_target               25                   2                  2     0.000037     0.000023  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000039             0.000019
7   recommendationservice      trace.to_productcatalogservice.row_count  to_target               25                   2                  2   590.600000   353.600000  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00           614.000000           180.000000
8                    root  trace.to_productcatalogservice.duration_mean  to_target               25                   2                  2     0.047740     0.003119  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.049377             0.002875
9         checkoutservice   trace.to_productcatalogservice.duration_p95  to_target               25                   2                  1     0.000038     0.000020  2024-02-20 23:40:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000045             0.000020
10        frontendservice   trace.to_productcatalogservice.duration_p95  to_target               25                   2                  1     0.000032     0.000024  2024-02-20 23:47:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00             0.000032             0.000023
11                   root      trace.to_productcatalogservice.row_count  to_target               25                   0                  2    24.000000    14.400000  2024-02-20 23:37:00+00:00 2024-02-20 23:37:00+00:00 2024-02-21 00:01:00+00:00            24.000000            12.000000, 16, 12)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' DataFrame if present; otherwise load it
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# 1) Parse timestamps as UTC
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Focus on productcatalogservice.diskio
svc = 'productcatalogservice'
kpi = 'diskio'
mask = (metric['cmdb_id'] == svc) & (metric['kpi_name'] == kpi)
pc_diskio_full = metric[mask].copy().sort_values('timestamp')

# 2) Compute global P95 on full series (before window filtering)
if pc_diskio_full.empty:
    raise RuntimeError("No metric data found for productcatalogservice.diskio in metric.csv")

global_p95 = float(pc_diskio_full['value'].quantile(0.95))

# 3) Filter to incident window
incident_start = pd.to_datetime("2024-02-20 23:34:32", utc=True)
incident_end   = pd.to_datetime("2024-02-21 00:04:32", utc=True)
pc_window = pc_diskio_full[(pc_diskio_full['timestamp'] >= incident_start) & (pc_diskio_full['timestamp'] <= incident_end)].copy().sort_values('timestamp')

total_count_in_window = int(pc_window.shape[0])

# 4) Identify consecutive in-window segments where value > global_p95
if pc_window.empty:
    segments_df = pd.DataFrame(columns=[
        'start_timestamp','end_timestamp','duration_minutes','count_in_segment',
        'max_value_in_segment','global_p95','absolute_deviation','relative_deviation_pct'
    ])
    earliest_segment_start = pd.NaT
else:
    pc_window['above'] = pc_window['value'] > global_p95
    above = pc_window[pc_window['above']].copy().sort_values('timestamp')
    if above.empty:
        segments_df = pd.DataFrame(columns=[
            'start_timestamp','end_timestamp','duration_minutes','count_in_segment',
            'max_value_in_segment','global_p95','absolute_deviation','relative_deviation_pct'
        ])
        earliest_segment_start = pd.NaT
    else:
        # Identify contiguous runs where consecutive timestamps differ by exactly 60 seconds
        diffs = above['timestamp'].diff().dt.total_seconds().fillna(0)
        new_group = (diffs != 60).cumsum()  # groups consecutive rows spaced by 60s into same group
        above = above.assign(group=new_group.values)
        segs = (
            above.groupby('group')
            .agg(
                start_timestamp=('timestamp','min'),
                end_timestamp=('timestamp','max'),
                count_in_segment=('timestamp','size'),
                max_value_in_segment=('value','max'),
                min_value_in_segment=('value','min')
            )
            .reset_index(drop=True)
        )
        segs['duration_minutes'] = segs['count_in_segment']  # each row is one minute
        segs['global_p95'] = global_p95
        segs['absolute_deviation'] = segs['max_value_in_segment'] - global_p95
        # Handle division by zero for relative deviation
        segs['relative_deviation_pct'] = np.where(
            global_p95 != 0,
            (segs['absolute_deviation'] / global_p95) * 100.0,
            np.nan
        )
        # Select requested columns and order
        segments_df = segs[
            [
                'start_timestamp','end_timestamp','duration_minutes','count_in_segment',
                'max_value_in_segment','global_p95','absolute_deviation','relative_deviation_pct'
            ]
        ].sort_values('start_timestamp').reset_index(drop=True)
        earliest_segment_start = segments_df['start_timestamp'].min()

# Keep outputs compact
segments_df, global_p95, total_count_in_window, earliest_segment_start

```
Out[8]:
```
```python
# Summarize the productcatalogservice.diskio segment analysis in plain English.
summary = (
    "productcatalogservice.diskio analysis (incident window 2024-02-20 23:34:32 to 2024-02-21 00:04:32 UTC):\n"
    f"- Global P95 (full series): {global_p95:.0f} (≈8.459e9).\n"
    f"- Total metric points in-window: {total_count_in_window}.\n"
    "- Identified consecutive segments where value > P95 (fault candidates): 2 segments (both single-minute spikes).\n"
    "  1) Start: 2024-02-20 23:51:00 UTC, End: 2024-02-20 23:51:00 UTC\n"
    f"     - Duration: 1 minute, count_in_segment: 1\n"
    f"     - Max value in segment: {segments_df.loc[0,'max_value_in_segment']:.0f}\n"
    f"     - Absolute deviation: {segments_df.loc[0,'absolute_deviation']:.0f}\n"
    f"     - Relative deviation: {segments_df.loc[0,'relative_deviation_pct']:.3f}%\n"
    "  2) Start: 2024-02-20 23:59:00 UTC, End: 2024-02-20 23:59:00 UTC\n"
    f"     - Duration: 1 minute, count_in_segment: 1\n"
    f"     - Max value in segment: {segments_df.loc[1,'max_value_in_segment']:.0f}\n"
    f"     - Absolute deviation: {segments_df.loc[1,'absolute_deviation']:.0f}\n"
    f"     - Relative deviation: {segments_df.loc[1,'relative_deviation_pct']:.3f}%\n"
    f"- Earliest segment start: {earliest_segment_start} UTC.\n\n"
    "Interpretation: diskio exceeded its global P95 only at two isolated minutes during the incident window (23:51 and 23:59 UTC). "
    "The larger spike at 23:59 shows a bigger absolute and relative deviation (~20.7 million above P95, ~0.245%), while the 23:51 spike is smaller (~7.6 million, ~0.090%). "
    "These are brief single-minute spikes rather than sustained high-disk-IO periods; investigate events or operations occurring at those exact minutes (e.g., bursts, backups, or I/O-heavy requests)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            start_timestamp             end_timestamp  duration_minutes  count_in_segment  max_value_in_segment    global_p95  absolute_deviation  relative_deviation_pct
0 2024-02-20 23:51:00+00:00 2024-02-20 23:51:00+00:00                 1                 1          8.466636e+09  8.459010e+09        7.626490e+06                0.090158
1 2024-02-20 23:59:00+00:00 2024-02-20 23:59:00+00:00                 1                 1          8.479708e+09  8.459010e+09        2.069823e+07                0.244689, 8459009911.532538, 25, Timestamp('2024-02-20 23:51:00+0000', tz='UTC'))```
```